# Pertemuan 6: Word Embedding dengan Skip-Gram

Notebook ini mendemonstrasikan cara merepresentasikan kata menjadi vektor menggunakan model Skip-Gram (Word2Vec).

**Skip-Gram** adalah model yang memprediksi kata-kata konteks (surrounding words) berdasarkan kata target.

**Mahasiswa:** Wahyu Pratama | **NPM:** 230411100058

## 1. Import Library

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from gensim.models import Word2Vec
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity
import warnings
warnings.filterwarnings('ignore')

# Konfigurasi visualisasi
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (12, 8)

print("Library berhasil dimuat")

## 2. Load Dataset 200 Berita

Memuat dataset berita finance dan sport untuk training Skip-Gram model.

In [ ]:
import re

# Load dataset
df_finance = pd.read_csv('detik_finance_100.csv')
df_sport = pd.read_csv('detik_sport_100.csv')

# Tambahkan label
df_finance['label'] = 'finance'
df_sport['label'] = 'sport'

# Gabungkan dataset
df = pd.concat([df_finance, df_sport], ignore_index=True)

print("=" * 60)
print("DATASET BERITA FINANCE & SPORT")
print("=" * 60)
print(f"Total dokumen: {len(df)}")
print(f"Finance: {len(df_finance)} berita")
print(f"Sport: {len(df_sport)} berita")
print(f"\nContoh berita pertama:")
print(f"Judul: {df.iloc[0]['judul']}")
print(f"Label: {df.iloc[0]['label']}")
print(f"Isi (100 char): {df.iloc[0]['isi_berita'][:100]}...")

## 3. Text Preprocessing

Membersihkan dan memproses teks untuk training Skip-Gram.

In [ ]:
def preprocess_text(text):
    """Preprocessing teks sederhana"""
    if pd.isna(text):
        return ""
    # Lowercase
    text = str(text).lower()
    # Hapus karakter khusus, hanya ambil huruf dan spasi
    text = re.sub(r'[^a-z\s]', ' ', text)
    # Hapus spasi berlebih
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Terapkan preprocessing
df['teks_bersih'] = df['isi_berita'].apply(preprocess_text)

print("=" * 60)
print("PREPROCESSING SELESAI")
print("=" * 60)
print(f"Contoh hasil preprocessing:")
print(f"\nOriginal: {df.iloc[0]['isi_berita'][:100]}...")
print(f"\nBersih: {df.iloc[0]['teks_bersih'][:100]}...")

# Tokenisasi - ubah setiap berita menjadi list kata
tokenized_sentences = [text.split() for text in df['teks_bersih'] if text]

print(f"\nTotal dokumen untuk training: {len(tokenized_sentences)}")
print(f"Contoh tokenized (dokumen 1, 10 kata pertama): {tokenized_sentences[0][:10]}")

# Statistik kata
all_words = [word for sentence in tokenized_sentences for word in sentence]
unique_words = sorted(set(all_words))

print(f"\nTotal kata: {len(all_words):,}")
print(f"Vocabulary size (kata unik): {len(unique_words):,}")
print(f"10 kata pertama dalam vocabulary: {unique_words[:10]}")

## 4. Training Skip-Gram Model

**Skip-Gram** adalah arsitektur Word2Vec yang:
- Input: kata target
- Output: kata-kata konteks di sekitarnya

**Parameter:**
- `vector_size`: dimensi vektor (100)
- `window`: ukuran jendela konteks (5 kata sebelum & sesudah)
- `min_count`: minimum frekuensi kata (2, buang kata yang terlalu jarang)
- `sg=1`: gunakan Skip-Gram (bukan CBOW)
- `epochs`: jumlah iterasi training (50)
- `workers`: parallel processing (4 threads)

In [ ]:
# Training Skip-Gram model dengan 200 dokumen
print("Training Skip-Gram model...")
print("Ini mungkin memakan waktu beberapa detik...\n")

model = Word2Vec(
    sentences=tokenized_sentences,
    vector_size=100,      # Dimensi vektor word embedding
    window=5,             # Context window size (lebih besar untuk dokumen panjang)
    min_count=2,          # Minimum word frequency (buang kata yang terlalu jarang)
    sg=1,                 # Skip-Gram (1) vs CBOW (0)
    epochs=50,            # Jumlah iterasi training
    workers=4,            # Parallel processing
    seed=42
)

print("=" * 60)
print("SKIP-GRAM MODEL BERHASIL DILATIH")
print("=" * 60)
print(f"Total dokumen training: {len(tokenized_sentences)}")
print(f"Vocabulary size: {len(model.wv)} kata")
print(f"Vector size: {model.wv.vector_size} dimensi")
print(f"Window size: {model.window}")
print(f"Training epochs: {model.epochs}")
print(f"\n10 kata pertama dalam vocabulary: {list(model.wv.index_to_key[:10])}")

## 5. Representasi Vektor Beberapa Kata

Menampilkan vektor representasi untuk kata-kata penting dari domain Finance dan Sport.

In [ ]:
# Pilih kata-kata penting untuk analisis
key_words_finance = ['bank', 'indonesia', 'rupiah', 'saham', 'ekonomi', 'harga']
key_words_sport = ['timnas', 'sepak', 'bola', 'pemain', 'pertandingan', 'gol']
key_words = key_words_finance + key_words_sport

print("=" * 60)
print("VEKTOR REPRESENTASI KATA-KATA PENTING")
print("=" * 60)

print("\nKata-kata Finance:")
for word in key_words_finance:
    if word in model.wv:
        vector = model.wv[word]
        print(f"\n  Kata: '{word}'")
        print(f"  Shape: {vector.shape}")
        print(f"  10 dimensi pertama: {vector[:10].round(4)}")
    else:
        print(f"\n  Kata: '{word}' (tidak ditemukan dalam vocabulary)")

print("\n" + "-" * 60)
print("Kata-kata Sport:")
for word in key_words_sport:
    if word in model.wv:
        vector = model.wv[word]
        print(f"\n  Kata: '{word}'")
        print(f"  Shape: {vector.shape}")
        print(f"  10 dimensi pertama: {vector[:10].round(4)}")
    else:
        print(f"\n  Kata: '{word}' (tidak ditemukan dalam vocabulary)")

## 6. Matriks Vektor (Sample 50 Kata Teratas)

Membuat DataFrame berisi vektor untuk 50 kata paling sering muncul.

In [ ]:
# Ambil 50 kata teratas (paling sering muncul)
top_50_words = list(model.wv.index_to_key[:50])

# Buat matriks vektor
word_vectors = {}
for word in top_50_words:
    word_vectors[word] = model.wv[word]

# Konversi ke DataFrame
df_vectors = pd.DataFrame(word_vectors).T
df_vectors.columns = [f'dim_{i+1}' for i in range(model.wv.vector_size)]

print("=" * 60)
print("MATRIKS VEKTOR (50 kata teratas, 10 dimensi pertama)")
print("=" * 60)
print(df_vectors.iloc[:, :10])

print(f"\nShape lengkap: {df_vectors.shape}")
print(f"(Jumlah kata: {df_vectors.shape[0]}, Dimensi vektor: {df_vectors.shape[1]})")

## 7. Cosine Similarity antar Kata

Mengukur kemiripan semantik antar kata menggunakan cosine similarity untuk kata-kata penting.

In [ ]:
# Pilih kata-kata untuk analisis similarity
selected_words = ['bank', 'indonesia', 'saham', 'ekonomi', 'timnas', 'pemain', 'pertandingan', 'gol']
selected_words = [w for w in selected_words if w in model.wv]  # Filter kata yang ada

if len(selected_words) > 0:
    # Ambil vektor kata-kata terpilih
    selected_vectors = np.array([model.wv[word] for word in selected_words])
    
    # Hitung cosine similarity
    similarity_matrix = cosine_similarity(selected_vectors)
    df_similarity = pd.DataFrame(
        similarity_matrix,
        index=selected_words,
        columns=selected_words
    )
    
    print("=" * 60)
    print("COSINE SIMILARITY MATRIX (Kata Terpilih)")
    print("=" * 60)
    print(df_similarity.round(4))
    
    # Visualisasi heatmap
    plt.figure(figsize=(10, 8))
    sns.heatmap(
        df_similarity,
        annot=True,
        fmt='.3f',
        cmap='RdYlGn',
        center=0.5,
        square=True,
        linewidths=0.5,
        cbar_kws={'label': 'Similarity'}
    )
    plt.title('Cosine Similarity Heatmap\n(Finance & Sport Keywords)', fontsize=14, fontweight='bold', pad=20)
    plt.xlabel('Kata', fontsize=12)
    plt.ylabel('Kata', fontsize=12)
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()
else:
    print("Kata-kata yang dipilih tidak ditemukan dalam vocabulary")

## 8. Kata-Kata Paling Mirip

Menggunakan fungsi `most_similar()` untuk mencari kata-kata yang semantically similar.

In [ ]:
print("=" * 60)
print("KATA-KATA PALING MIRIP (TOP 5)")
print("=" * 60)

# Pilih kata kunci untuk analisis
key_words_to_analyze = [
    'bank', 'indonesia', 'ekonomi', 'saham',  # Finance
    'timnas', 'pemain', 'pertandingan', 'sepak'  # Sport
]

for word in key_words_to_analyze:
    if word in model.wv:
        print(f"\nKata: '{word}'")
        try:
            similar_words = model.wv.most_similar(word, topn=5)
            for i, (similar_word, score) in enumerate(similar_words, 1):
                print(f"  {i}. {similar_word:.<20} (similarity: {score:.4f})")
        except:
            print(f"  (tidak dapat menghitung similarity)")
    else:
        print(f"\nKata: '{word}' (tidak ditemukan dalam vocabulary)")

## 9. Visualisasi 2D dengan PCA

Mereduksi vektor 100 dimensi menjadi 2 dimensi untuk visualisasi. Menampilkan kata-kata Finance dan Sport.

In [ ]:
# Pilih kata-kata representatif untuk visualisasi
finance_words = [w for w in ['bank', 'indonesia', 'rupiah', 'saham', 'ekonomi', 'harga', 'keuangan', 'investasi'] if w in model.wv]
sport_words = [w for w in ['timnas', 'sepak', 'bola', 'pemain', 'pertandingan', 'gol', 'liga', 'juara'] if w in model.wv]

# Gabungkan
viz_words = finance_words + sport_words

if len(viz_words) > 0:
    # Ambil vektor
    viz_vectors = np.array([model.wv[word] for word in viz_words])
    
    # PCA untuk reduksi dimensi ke 2D
    pca = PCA(n_components=2, random_state=42)
    vectors_2d = pca.fit_transform(viz_vectors)
    
    # Buat DataFrame untuk plotting
    df_plot = pd.DataFrame({
        'word': viz_words,
        'x': vectors_2d[:, 0],
        'y': vectors_2d[:, 1],
        'category': ['Finance']*len(finance_words) + ['Sport']*len(sport_words)
    })
    
    print("=" * 60)
    print("KOORDINAT 2D SETELAH PCA")
    print("=" * 60)
    print(df_plot)
    
    # Visualisasi scatter plot
    plt.figure(figsize=(14, 10))
    
    # Plot Finance words
    finance_df = df_plot[df_plot['category'] == 'Finance']
    plt.scatter(finance_df['x'], finance_df['y'], 
               s=300, alpha=0.6, c='steelblue', edgecolors='black', linewidth=1.5, label='Finance')
    
    # Plot Sport words
    sport_df = df_plot[df_plot['category'] == 'Sport']
    plt.scatter(sport_df['x'], sport_df['y'], 
               s=300, alpha=0.6, c='coral', edgecolors='black', linewidth=1.5, label='Sport')
    
    # Annotate each point
    for idx, row in df_plot.iterrows():
        plt.annotate(
            row['word'],
            (row['x'], row['y']),
            fontsize=11,
            fontweight='bold',
            ha='center',
            va='bottom',
            xytext=(0, 5),
            textcoords='offset points'
        )
    
    plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)', fontsize=12)
    plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)', fontsize=12)
    plt.title('Visualisasi 2D Word Embeddings (Skip-Gram)\n200 Dokumen Berita Finance & Sport', 
             fontsize=14, fontweight='bold')
    plt.legend(fontsize=12, loc='best')
    plt.grid(alpha=0.3, linestyle='--')
    plt.axhline(y=0, color='gray', linewidth=0.8, alpha=0.5)
    plt.axvline(x=0, color='gray', linewidth=0.8, alpha=0.5)
    plt.tight_layout()
    plt.show()
    
    print(f"\nVariance explained by 2 components: {pca.explained_variance_ratio_.sum():.2%}")
else:
    print("Tidak ada kata yang ditemukan untuk visualisasi")

## 10. Operasi Aritmatika Vektor

Word embeddings memungkinkan operasi matematika pada vektor kata.

In [ ]:
print("=" * 60)
print("OPERASI ARITMATIKA VEKTOR")
print("=" * 60)

# Contoh 1: bank + indonesia
if 'bank' in model.wv and 'indonesia' in model.wv:
    result_vector = model.wv['bank'] + model.wv['indonesia']
    print("\nOperasi: bank + indonesia")
    print(f"Hasil vektor shape: {result_vector.shape}")
    
    try:
        similar = model.wv.similar_by_vector(result_vector, topn=5)
        print("Kata paling mirip dengan hasil 'bank + indonesia':")
        for word, score in similar:
            print(f"  - {word:.<20} (similarity: {score:.4f})")
    except:
        print("  (tidak dapat menghitung similarity)")

# Contoh 2: timnas - indonesia
if 'timnas' in model.wv and 'indonesia' in model.wv:
    result_vector = model.wv['timnas'] - model.wv['indonesia']
    print("\n" + "-" * 60)
    print("Operasi: timnas - indonesia")
    print(f"Hasil vektor shape: {result_vector.shape}")
    
    try:
        similar = model.wv.similar_by_vector(result_vector, topn=5)
        print("Kata paling mirip dengan hasil 'timnas - indonesia':")
        for word, score in similar:
            print(f"  - {word:.<20} (similarity: {score:.4f})")
    except:
        print("  (tidak dapat menghitung similarity)")

# Contoh 3: pemain + bola
if 'pemain' in model.wv and 'bola' in model.wv:
    result_vector = model.wv['pemain'] + model.wv['bola']
    print("\n" + "-" * 60)
    print("Operasi: pemain + bola")
    print(f"Hasil vektor shape: {result_vector.shape}")
    
    try:
        similar = model.wv.similar_by_vector(result_vector, topn=5)
        print("Kata paling mirip dengan hasil 'pemain + bola':")
        for word, score in similar:
            print(f"  - {word:.<20} (similarity: {score:.4f})")
    except:
        print("  (tidak dapat menghitung similarity)")

## 11. Analisis Context Window

Menunjukkan bagaimana Skip-Gram menggunakan context window untuk belajar.

In [ ]:
print("=" * 60)
print("CONTOH CONTEXT WINDOW (window=5)")
print("=" * 60)

# Ambil dokumen pertama sebagai contoh
example_sentence = tokenized_sentences[0][:20]  # 20 kata pertama
print(f"\nDokumen contoh (20 kata pertama): {' '.join(example_sentence)}")
print("\nSkip-Gram training pairs (target -> context):")

window_size = 5
# Tampilkan hanya 5 kata pertama sebagai contoh
for i in range(min(5, len(example_sentence))):
    target = example_sentence[i]
    # Tentukan context words (window size = 5)
    start = max(0, i - window_size)
    end = min(len(example_sentence), i + window_size + 1)
    context = [example_sentence[j] for j in range(start, end) if j != i]
    
    print(f"\nTarget: '{target}'")
    print(f"Context words: {context[:10]}")  # Tampilkan max 10 context words
    print(f"Jumlah training pairs: {len(context)}")

## 12. Ringkasan & Kesimpulan

**Skip-Gram Word2Vec** berhasil merepresentasikan setiap kata dari 200 dokumen berita menjadi vektor numerik yang menangkap makna semantik kata.

**Dataset:**
- 200 dokumen berita (100 Finance + 100 Sport)
- Vocabulary size: ribuan kata unik
- Vector size: 100 dimensi per kata

**Hasil:**
- Kata-kata Finance ('bank', 'ekonomi', 'saham') memiliki similarity tinggi satu sama lain
- Kata-kata Sport ('timnas', 'pemain', 'pertandingan') juga berkelompok
- Model dapat melakukan operasi aritmatika vektor
- Visualisasi 2D menunjukkan pemisahan semantik antara domain Finance dan Sport

**Keunggulan:**
- Menangkap hubungan semantik antar kata
- Mendukung operasi matematika pada vektor
- Dapat divisualisasikan dalam ruang berdimensi rendah
- Useful untuk downstream tasks (klasifikasi, clustering, dll)

In [ ]:
print("=" * 60)
print("RINGKASAN SKIP-GRAM MODEL")
print("=" * 60)

summary = {
    'Jumlah Dokumen': len(df),
    'Dokumen Finance': len(df_finance),
    'Dokumen Sport': len(df_sport),
    'Total Kata (Tokens)': f"{len(all_words):,}",
    'Vocabulary Size': f"{len(model.wv):,} kata",
    'Vector Dimensi': model.wv.vector_size,
    'Window Size': model.window,
    'Min Count': 2,
    'Training Epochs': model.epochs,
    'Model Type': 'Skip-Gram (sg=1)'
}

for key, value in summary.items():
    print(f"{key:.<45} {value}")

print("\n" + "=" * 60)
print("APLIKASI WORD EMBEDDINGS")
print("=" * 60)
print("1. Text Classification (Finance vs Sport)")
print("2. Document Similarity & Clustering")
print("3. Information Retrieval & Search")
print("4. Named Entity Recognition")
print("5. Sentiment Analysis")
print("6. Machine Translation")

print("\n" + "=" * 60)
print("Mahasiswa: Wahyu Pratama | NPM: 230411100058")
print("Pertemuan 6: Skip-Gram Word2Vec")
print("=" * 60)